# all-reduce-compose — faded example 2: Add the missing broadcast leg of all_reduce

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-compose`. Running the beacon reports progress on the `Distributed: all_reduce composition` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce composition` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-compose`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-compose"
DD_SUBTOPIC = "Distributed: all_reduce composition"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A `reduce(dst=0)` alone is NOT an all_reduce: only rank 0 ends up with the aggregated value while the other ranks are stale. The defining second leg is `broadcast(src=0)`, which copies rank 0's result back to every rank. Without it the collective is merely a reduce.

## Faded exercise 2

### Faded — supply the broadcast that makes it an all_reduce

`ex_all_reduce_sum(rank, world_size, dist_module, local_value)` already does the SUM reduce onto rank 0. As written, non-zero ranks would return a stale value. Add the one collective that turns this reduce into a true all_reduce so every rank returns the global sum.

**Fill in:** The broadcast call from rank 0 that copies the reduced sum back to every rank.

In [ ]:
import threading

class MockDist:
    class ReduceOp:
        SUM = 'sum'
    def __init__(self, world_size):
        self.world_size = world_size
        self._barrier = threading.Barrier(world_size)
        self._slots = [None] * world_size
        self._result = [None]
    def reduce(self, tensor, dst, op):
        rank = threading.current_thread().rank
        self._slots[rank] = tensor.clone()
        self._barrier.wait()
        if rank == dst:
            tensor.copy_(t.stack(self._slots).sum(dim=0))
            self._result[0] = tensor.clone()
        self._barrier.wait()
    def broadcast(self, tensor, src):
        self._barrier.wait()
        tensor.copy_(self._result[0])
        self._barrier.wait()

def ex_all_reduce_sum(rank: int, world_size: int, dist_module, local_value: float) -> float:
    tensor = t.tensor([local_value], dtype=t.float32)
    dist_module.reduce(tensor, dst=0, op=dist_module.ReduceOp.SUM)
    dist_module.broadcast(tensor, src=0)
    return tensor.item()

def _test():
    world_size = 3
    locals_ = [2.0, 4.0, 8.0]
    mock = MockDist(world_size)
    results = [None] * world_size
    def _run(rank):
        threading.current_thread().rank = rank
        results[rank] = ex_all_reduce_sum(rank, world_size, mock, locals_[rank])
    threads = [threading.Thread(target=_run, args=(r,)) for r in range(world_size)]
    for th in threads: th.start()
    for th in threads: th.join()
    expected = sum(locals_)
    assert len(set(results)) == 1, f'ranks disagree (broadcast missing?): {results}'
    assert abs(results[0] - expected) < 1e-6, f'got {results[0]}, expected {expected}'

try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import threading

class MockDist:
    class ReduceOp:
        SUM = 'sum'
    def __init__(self, world_size):
        self.world_size = world_size
        self._barrier = threading.Barrier(world_size)
        self._slots = [None] * world_size
        self._result = [None]
    def reduce(self, tensor, dst, op):
        rank = threading.current_thread().rank
        self._slots[rank] = tensor.clone()
        self._barrier.wait()
        if rank == dst:
            tensor.copy_(t.stack(self._slots).sum(dim=0))
            self._result[0] = tensor.clone()
        self._barrier.wait()
    def broadcast(self, tensor, src):
        self._barrier.wait()
        tensor.copy_(self._result[0])
        self._barrier.wait()

def ex_all_reduce_sum(rank: int, world_size: int, dist_module, local_value: float) -> float:
    tensor = t.tensor([local_value], dtype=t.float32)
    dist_module.reduce(tensor, dst=0, op=dist_module.ReduceOp.SUM)
    dist_module.broadcast(tensor, src=0)
    return tensor.item()
```
</details>